# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/Lee-Yongsu/2026-HYU-AUE8088-PA2.git
else:
    !git pull origin main

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 36 (delta 3), reused 0 (delta 0), pack-reused 23 (from 1)
Receiving objects: 100% (36/36), 47.09 KiB | 9.42 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 63.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 22.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss, LDAMLoss, weighted_cross_entropy
from src.augment.mix import mixup_data, cutmix_data, mixed_loss
from src.models.vit import vit_small_patch16_224, load_pretrained_vit_s16

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level3"]
# 각 실험의 run 이름은 run_experiment(name=...) 가 부여하여 동일 프로젝트에 누적된다.

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

# val_loader 는 모든 실험에서 동일
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# 속성별 클래스 분포 — 소수 클래스(snowy/foggy/dawn-dusk) 불균형 확인
for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

In [ ]:
from tqdm import tqdm
import os, urllib.request

# --- ImageNet(DeiT-S/16) pretrained 체크포인트 한 번만 다운로드 ---
CKPT_URL  = "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth"
CKPT_PATH = "../checkpoints/deit_small_patch16_224.pth"
os.makedirs("../checkpoints", exist_ok=True)
if not os.path.exists(CKPT_PATH):
    print("DeiT-S/16 체크포인트 다운로드 중...")
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)


def build_model():
    """ViT-S/16 생성 + ImageNet(DeiT) pretrained 로드. 매 run 동일 init 보장 위해 seed 고정."""
    set_seed(SEED, deterministic=True)
    m = vit_small_patch16_224().to(device)
    pre = torch.load(CKPT_PATH, map_location="cpu")
    load_pretrained_vit_s16(m, pre)
    return m


def build_loss_fns(kind):
    """kind='ce'        -> baseline 일반 CrossEntropy (불균형 대응 없음)
       kind='imbalance' -> 속성별 imbalance-aware loss (Focal/CB/CE)"""
    if kind == "ce":
        return {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    return {
        "weather":   FocalLoss(gamma=2.0).to(device),                      # 최다 불균형 → Focal
        "scene":     ClassBalancedLoss(train_ds.class_counts("scene")).to(device),
        "timeofday": nn.CrossEntropyLoss(),
    }


def loss_desc(kind):
    """wandb config 에 사람이 읽을 수 있는 속성별 loss 이름을 남기기 위한 설명."""
    if kind == "ce":
        return {a: "ce" for a in ATTRIBUTES}
    return {"weather": "focal_g2.0", "scene": "cb_loss", "timeofday": "ce"}


def download_to_local(path):
    """Colab 환경이면 체크포인트를 브라우저로 로컬 다운로드. 그 외 환경에선 조용히 skip."""
    try:
        from google.colab import files
        files.download(path)
    except Exception as e:
        print(f"[local download skipped] {e}")


class MixupTrainer(MultiTaskTrainer):
    """use_mix=True 이면 매 step 50% Mixup / 50% CutMix 를 적용 (그 외엔 부모와 동일)."""

    def __init__(self, *args, use_mix=False, **kwargs):
        super().__init__(*args, **kwargs)
        self.use_mix = use_mix

    def _train_one_epoch(self, loader, epoch):
        if not self.use_mix:
            return super()._train_one_epoch(loader, epoch)

        self.model.train()
        running, n = 0.0, 0
        pbar = tqdm(loader, desc=f"train e{epoch+1}", leave=False)
        for batch in pbar:
            x = batch["image"].to(self.device, non_blocking=True)
            y = {a: batch[a].to(self.device, non_blocking=True) for a in ATTRIBUTES}

            self.optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", enabled=self.cfg.amp):
                if torch.rand(1).item() < 0.5:
                    xm, ya, yb, lam = mixup_data(x, y, alpha=0.2)
                else:
                    xm, ya, yb, lam = cutmix_data(x, y, alpha=1.0)
                logits = self.model(xm)
                loss = mixed_loss(self.loss_fns, logits, ya, yb, lam, weights=self.cfg.loss_weights)

            self.scaler.scale(loss).backward()
            if self.cfg.grad_clip is not None:
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            running += loss.item(); n += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        return running / max(n, 1)


EPOCHS = 25


def run_experiment(name, *, use_sampler, use_mix, loss_kind="imbalance", epochs=EPOCHS):
    """단일 불균형-대응 설정을 학습 → wandb 로깅 + 체크포인트 저장 + per-class F1 반환.

    모든 run 은 동일 seed/epochs/lr, 매번 ImageNet pretrained 에서 시작하여 공정 비교.
    """
    set_seed(SEED, deterministic=True)
    model = build_model()
    loss_fns = build_loss_fns(loss_kind)

    # train_loader — sampler 사용 여부에 따라 구성
    g = torch.Generator(); g.manual_seed(SEED)
    if use_sampler:
        sampler = class_balanced_sampler(train_ds, attribute="weather")
        train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                                  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                                  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)

    # ViT pretrained fine-tune 용 하이퍼파라미터
    optim = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level3-{name}",
        config={
            "backbone": "vit_s16_pretrained", "exp": name,
            "use_sampler": use_sampler, "use_mix": use_mix, "loss_kind": loss_kind,
            "sampler": "class_balanced(weather)" if use_sampler else "none",
            "loss": loss_desc(loss_kind),
            "epochs": epochs, "batch": BATCH, "lr": 1e-4, "weight_decay": 5e-2, "seed": SEED,
        },
        tags=WANDB_TAGS + [name],
    )
    trainer = MixupTrainer(model, optim, sched, loss_fns, device,
                           TrainConfig(epochs=epochs), logger=logger, use_mix=use_mix)
    history = trainer.fit(train_loader, val_loader)

    # 최종 모델 기준 — confusion matrix + per-class P/R/F1 로깅
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    prf = per_class_prf(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
        rows = list(zip(prf[a]["class"], prf[a]["precision"], prf[a]["recall"], prf[a]["f1"], prf[a]["support"]))
        logger.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"], [list(r) for r in rows])
    logger.finish()

    safe = name.replace("+", "_")
    ckpt_path = f"../checkpoints/level3_{safe}.pth"
    torch.save(model.state_dict(), ckpt_path)
    download_to_local(ckpt_path)   # 학습 종료 후 로컬로 다운로드
    return {"name": name, "history": history, "prf": prf, "cms": cms,
            "avg_mf1": history["val_avg_mf1"][-1]}

In [ ]:
# 커리큘럼 — 변수를 하나씩만 바꾸는 ablation (5 runs).
#   R0 baseline        : 불균형 대응 전혀 없음            (분석 ①②의 "before" 기준선)
#   R1 loss            : 속성별 imbalance-aware loss
#   R2 loss+sampler    : + class-balanced sampler
#   R3 loss+mix        : + Mixup/CutMix
#   R4 loss+sampler+mix: 전체                              (분석 ①②의 "after")
# R1~R4 는 loss 고정 2x2 → Sampling x Mixup 상호작용 분석 (분석 ③).
results = {}
results["baseline"]         = run_experiment("baseline",         use_sampler=False, use_mix=False, loss_kind="ce")
results["loss"]             = run_experiment("loss",             use_sampler=False, use_mix=False)
results["loss+sampler"]     = run_experiment("loss+sampler",     use_sampler=True,  use_mix=False)
# results["loss+mix"]         = run_experiment("loss+mix",         use_sampler=False, use_mix=True)
# results["loss+sampler+mix"] = run_experiment("loss+sampler+mix", use_sampler=True,  use_mix=True)

In [ ]:
results["loss+mix"]         = run_experiment("loss+mix",         use_sampler=False, use_mix=True)
results["loss+sampler+mix"] = run_experiment("loss+sampler+mix", use_sampler=True,  use_mix=True)

In [ ]:
# 분석용 비교 표 — 소수/다수 클래스 per-class F1 + Avg-MF1 을 run 별로 정리
import pandas as pd

MINORITY = {"weather": ["snowy", "foggy"], "timeofday": ["dawn/dusk"]}  # 소수 클래스
MAJORITY = {"weather": ["clear"],          "timeofday": ["daytime"]}    # 다수 클래스

def f1_of(res, attr, cls):
    d = dict(zip(res["prf"][attr]["class"], res["prf"][attr]["f1"]))
    return d.get(cls, float("nan"))

rows = []
for name, res in results.items():
    row = {"run": name, "avg_mf1": round(res["avg_mf1"], 4)}
    for attr, classes in MINORITY.items():
        for c in classes:
            row[f"min:{attr}/{c}"] = round(f1_of(res, attr, c), 4)
    for attr, classes in MAJORITY.items():
        for c in classes:
            row[f"maj:{attr}/{c}"] = round(f1_of(res, attr, c), 4)
    rows.append(row)

df = pd.DataFrame(rows).set_index("run")
print("=== 분석 ①② : 소수(min) ↑ / 다수(maj) 회귀 / Avg-MF1 ===")
print(df, "\n")

# 분석 ③ : Sampling x Mixup 상호작용 (loss 고정 2x2, Avg-MF1 기준)
#   interaction = (both - sampler_only) - (mix_only - neither)
inter = (results["loss+sampler+mix"]["avg_mf1"] - results["loss+sampler"]["avg_mf1"]) \
      - (results["loss+mix"]["avg_mf1"]         - results["loss"]["avg_mf1"])
print("=== 분석 ③ : Sampling x Mixup 상호작용 ===")
print(f"interaction on Avg-MF1 = {inter:+.4f}  → {'시너지 (서로 도움)' if inter > 0 else '충돌 (간섭)'}")
df

## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.